
<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 2</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Modify Data with INSERT, UPDATE, and DELETE</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Add, change, and remove rows in a Delta table using standard SQL operations.</div>
</div>

</div>

**Where we left off:** In Lesson 1, we created an `employees` Delta table from a CSV file. It has 4 rows from the original data in the file.

In [0]:
%sql
USE CATALOG dbacademy;
USE SCHEMA get_started_de;

SELECT * FROM employees;

ID,FirstName,Country,Role,_rescued_data
1,Kristi,United States,Data Engineer,null
2,Sophia,Germany,Data Analyst,null
3,Peter,Canada,ML Engineer,null
4,Zebi,India,Data Scientist,null



<!-- LEARN: DML Operations on Delta Tables -->
<!-- Template: 3-card-centered-text -->

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">Delta Tables Support Full SQL Modifications</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Unlike plain data files, Delta tables let you add, change, and remove individual rows just like a traditional database. Every change is recorded as a new version of the table.</div>

<div style="display: flex; justify-content: center; align-items: stretch; gap: 40px; flex-wrap: wrap;">

  <!-- Card 1: INSERT -->
  <div style="width: 300px; min-height: 200px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); display: flex; flex-direction: column; justify-content: center; gap: 10px; padding: 24px; text-align: center; position: relative; box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 8px; background: #4299E0; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 18pt; font-weight: 600; color: #0b2026;">INSERT</div>
    <div style="font-size: 14pt; color: #5E7077;">Add new rows to a table. The table grows, and a new version is created.</div>
  </div>

  <!-- Card 2: UPDATE -->
  <div style="width: 300px; min-height: 200px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); display: flex; flex-direction: column; justify-content: center; gap: 10px; padding: 24px; text-align: center; position: relative; box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 8px; background: #00A972; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 18pt; font-weight: 600; color: #0b2026;">UPDATE</div>
    <div style="font-size: 14pt; color: #5E7077;">Change values in existing rows that match a condition. A new version is created.</div>
  </div>

  <!-- Card 3: DELETE -->
  <div style="width: 300px; min-height: 200px; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); display: flex; flex-direction: column; justify-content: center; gap: 10px; padding: 24px; text-align: center; position: relative; box-sizing: border-box;">
    <div style="position: absolute; top: 0; left: 0; width: 100%; height: 8px; background: #FF5F46; border-radius: 8px 8px 0 0;"></div>
    <div style="font-size: 18pt; font-weight: 600; color: #0b2026;">DELETE</div>
    <div style="font-size: 14pt; color: #5E7077;">Remove rows that match a condition. The data is gone from the current version, but still recoverable.</div>
  </div>

</div>

<!-- Key point callout -->
<div style="margin-top: 20px; padding: 16px 20px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Key concept:</strong> Every INSERT, UPDATE, and DELETE creates a new version of the table. Delta Lake tracks all of these versions automatically, which means you can always see what your data looked like before a change (see Lesson 3 to learn how to explore these versions). 
  </div>
</div>

</div>

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

##### EXPAND FOR ADDITIONAL NOTES

<details>

**Why this matters for data engineering**

- Traditional data lakes built on plain files (CSV, Parquet) don't support row-level changes. If you need to update a record, you have to rewrite the entire file. Delta Lake solves this by adding a transaction log on top of your data files.
- Each DML operation (INSERT, UPDATE, DELETE) is **atomic**: it either fully succeeds or fully rolls back. You won't end up with half-written changes.
- These operations use standard SQL syntax. If you've worked with databases like PostgreSQL, MySQL, or SQL Server, the commands are the same.
- Under the hood, Delta Lake doesn't modify existing files. It writes new data files and updates the transaction log to point to the latest version. This is what makes versioning and time travel possible.

</details>

</div>

### Explore: Insert new rows

Let's add two new employees to the table.

In [0]:
%sql
INSERT INTO employees (ID, FirstName, Country, Role) VALUES
  (5, 'Maria', 'Brazil', 'Data Engineer'),
  (6, 'Aiden', 'Australia', 'Data Analyst');

num_affected_rows,num_inserted_rows
2,2


Query the table to confirm the new rows were added.

In [0]:
%sql
SELECT * FROM employees;

ID,FirstName,Country,Role,_rescued_data
1,Kristi,United States,Data Engineer,null
2,Sophia,Germany,Data Analyst,null
3,Peter,Canada,ML Engineer,null
4,Zebi,India,Data Scientist,null
5,Maria,Brazil,Data Engineer,null
6,Aiden,Australia,Data Analyst,null
5,Maria,Brazil,Data Engineer,null
6,Aiden,Australia,Data Analyst,null


You should now see **6 rows** — the original 4 plus the 2 you just inserted.

### Explore: Update an existing row

Maria just moved to a new role. Let's update her record.

In [0]:
%sql
UPDATE employees
SET Role = 'Senior Data Engineer'
WHERE FirstName = 'Maria';

num_affected_rows
2


In [0]:
%sql
SELECT * FROM employees WHERE FirstName = 'Maria';

ID,FirstName,Country,Role,_rescued_data
5,Maria,Brazil,Senior Data Engineer,null
5,Maria,Brazil,Senior Data Engineer,null


Maria's role changed from `Data Engineer` to `Senior Data Engineer`. The original value isn't lost — Delta Lake stored it in a previous version of the table.

### Explore: Delete a row

Aiden has left the company. Let's remove his record.

In [0]:
%sql
DELETE FROM employees
WHERE FirstName = 'Aiden';

num_affected_rows
2


In [0]:
%sql
SELECT * FROM employees;

ID,FirstName,Country,Role,_rescued_data
1,Kristi,United States,Data Engineer,null
2,Sophia,Germany,Data Analyst,null
3,Peter,Canada,ML Engineer,null
4,Zebi,India,Data Scientist,null
5,Maria,Brazil,Senior Data Engineer,null
5,Maria,Brazil,Senior Data Engineer,null


The table is back to **5 rows**. Aiden's record is gone from the current version, but because Delta Lake tracks every change, it's still in the table's version history.

### Explore: Check the version history

Every operation you just ran created a new version of the table. Let's verify.

In [0]:
%sql
DESCRIBE HISTORY employees;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-10-04T09:50:56.000Z,73992968705957,nodirbeksadullayev512@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2337613719152859),ea69c119-276c-496c-8b0b-97ee5b7e28ce,1004-094856-u4b1ogce-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2006, p25FileSize -> 1924, numDeletionVectorsRemoved -> 1, minFileSize -> 1924, numAddedFiles -> 1, maxFileSize -> 1924, p75FileSize -> 1924, p50FileSize -> 1924, numAddedBytes -> 1924)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
5,2026-10-04T09:50:54.000Z,73992968705957,nodirbeksadullayev512@outlook.com,DELETE,"Map(predicate -> [""(FirstName#11582 = Aiden)""])",null,List(2337613719152859),ea69c119-276c-496c-8b0b-97ee5b7e28ce,1004-094856-u4b1ogce-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1958, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 1326, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 624)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
4,2026-10-04T09:50:35.000Z,73992968705957,nodirbeksadullayev512@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2337613719152859),eecf51ff-273c-4e0c-81ac-9cceaa61aeca,1004-094856-u4b1ogce-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 7069, p25FileSize -> 2006, numDeletionVectorsRemoved -> 2, minFileSize -> 2006, numAddedFiles -> 1, maxFileSize -> 2006, p75FileSize -> 2006, p50FileSize -> 2006, numAddedBytes -> 2006)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
3,2026-10-04T09:50:32.000Z,73992968705957,nodirbeksadullayev512@outlook.com,UPDATE,"Map(predicate -> [""(FirstName#10903 = Maria)""])",null,List(2337613719152859),eecf51ff-273c-4e0c-81ac-9cceaa61aeca,1004-094856-u4b1ogce-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5192, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2438, numAddedFiles -> 1, numUpdatedRows -> 2, numAddedBytes -> 1795, rewriteTimeMs -> 2695)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
2,2026-10-04T09:50:04.000Z,73992968705957,nodirbeksadullayev512@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2337613719152859),702151ec-0f71-4f11-8281-306b69f52b27,1004-094856-u4b1ogce-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1722)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
1,2026-10-04T09:49:41.000Z,73992968705957,nodirbeksadullayev512@outlook.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2337613719152859),aed55192-4337-4dcb-9d92-2fc9df33f346,1004-094856-u4b1ogce-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1722)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
0,2026-08-13T11:22:39.000Z,73992968705957,nodirbeksadullayev512@outlook.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2337613719152862),50e70693-fdc2-462a-b1f5-b5b4ce6e6661,0813-112038-8b36czao-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1830)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


You should see multiple versions:
- **Version 0** — `CREATE TABLE` (from Lesson 1)
- **Version 1** — `WRITE` (the INSERT)
- **Version 2** — `UPDATE`
- **Version 3** — `DELETE`

Each version records who made the change, when, and what type of operation it was. You'll learn how to query these older versions in Lesson 3.


<!-- Micro-win summary -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">What you just did:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Added rows with <code>INSERT INTO</code></li>
      <li>Changed a value with <code>UPDATE ... SET ... WHERE</code></li>
      <li>Removed a row with <code>DELETE FROM ... WHERE</code></li>
      <li>Verified all changes are tracked with <code>DESCRIBE HISTORY</code></li>
    </ul>
  </div>
</div>
</div>